# 🧵 スパースモデリングとグループラッソ入門

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Murakami-Shoya/study-notebooks/blob/main/notebooks/2026-10-20_group-lasso.ipynb)

**出典**: [Notion教材ページ](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea)（理論・数式・用語解説はNotionを参照）

**日付**: 2026-10-20

In [ ]:
# セットアップ（ローカル・Colab両対応）
import sys

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    pass

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local'}")

## シナリオ一覧

Notionページの主張をコードで確認する．

| # | シナリオ（何を動かすか） | 確かめること | 期待する結果 | 解説の節 |
|---|---|---|---|---|
| S1 | Lasso vs Ridge正則化経路 | Lassoは係数を0にするがRidgeは0にしない | Lasso係数は途中で0，Ridgeは0に近づくが0にならない | [§2 Lasso](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-1) |
| S2 | Group Lasso実装（ISTA） | ブロック軟閾値でグループごと0になる | 電極0,1だけ残り，他8電極は全係数0 | [§3 グループラッソ](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-2), [§5 解き方](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-4) |
| S3 | Lasso vs Group Lasso経路比較 | Group Lassoは同じ電極の係数が同時に入る | 経路プロットで同色（同電極）の線が同じλで立ち上がる | [§3 グループラッソ](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-2) |
| S4 | 交差検証でλ選択 | CV MSEが最小になるλを見つける | λ候補の中で検証誤差最小点が存在 | [§6 λの選び方](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-5) |

## 準備：データ生成とヘルパー関数

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Lasso, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

In [ ]:
def make_simple_data(n=100, p=10, active_idx=[0, 2, 5], seed=42):
    """Simple synthetic regression data."""
    rng = np.random.default_rng(seed)
    X = rng.standard_normal((n, p))
    w_true = np.zeros(p)
    w_true[active_idx] = [2.0, -1.5, 1.0]
    y = X @ w_true + 0.5 * rng.standard_normal(n)
    return X, y, w_true

In [ ]:
def make_grouped_data(n=200, n_elec=10, n_bands=4, active_elec=[0, 1], seed=0):
    """Synthetic data with electrode x band structure."""
    rng = np.random.default_rng(seed)
    p = n_elec * n_bands
    X = rng.standard_normal((n, p))
    w_true = np.zeros(p)
    for e in active_elec:
        w_true[e * n_bands:(e + 1) * n_bands] = rng.uniform(-1, 1, n_bands)
    y = X @ w_true + 0.5 * rng.standard_normal(n)
    groups = np.repeat(np.arange(n_elec), n_bands)
    return X, y, w_true, groups

In [ ]:
def group_lasso_ista(X, y, groups, lam, n_iter=500, tol=1e-6):
    """Group Lasso via ISTA (proximal gradient)."""
    n, p = X.shape
    w = np.zeros(p)
    L = np.linalg.norm(X, ord=2) ** 2 / n
    eta = 1.0 / L
    for _ in range(n_iter):
        w_old = w.copy()
        grad = X.T @ (X @ w - y) / n
        z = w - eta * grad
        for g in np.unique(groups):
            idx = groups == g
            p_g = idx.sum()
            norm_z_g = np.linalg.norm(z[idx])
            threshold = lam * np.sqrt(p_g) * eta
            if norm_z_g > threshold:
                w[idx] = (1 - threshold / norm_z_g) * z[idx]
            else:
                w[idx] = 0.0
        if np.linalg.norm(w - w_old) < tol:
            break
    return w

In [ ]:
def active_groups(w, groups, threshold=1e-8):
    """Return list of active group IDs."""
    return [g for g in np.unique(groups) if np.any(np.abs(w[groups == g]) > threshold)]

## S1. Lasso vs Ridge正則化経路

Lassoは係数をちょうど0にし，Ridgeは0に近づけるが0にはしない（[Notion §2](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-1)）．10特徴中3つが効くデータで確認する．

In [ ]:
X, y, _ = make_simple_data()
X_scaled = StandardScaler().fit_transform(X)

In [ ]:
alphas = np.logspace(-3, 1, 50)
lasso_coefs = np.array([Lasso(alpha=a, max_iter=10000).fit(X_scaled, y).coef_ for a in alphas])
ridge_coefs = np.array([Ridge(alpha=a).fit(X_scaled, y).coef_ for a in alphas])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for i in range(10):
    axes[0].plot(alphas, lasso_coefs[:, i], alpha=0.7, linewidth=1)
    axes[1].plot(alphas, ridge_coefs[:, i], alpha=0.7, linewidth=1)
for ax in axes:
    ax.set_xscale('log')
    ax.axhline(0, color='k', linestyle='--', linewidth=0.5)
    ax.set_xlabel('lambda')
    ax.set_ylabel('Coefficient')
    ax.grid(True, alpha=0.3)
axes[0].set_title('Lasso')
axes[1].set_title('Ridge')
plt.tight_layout()
plt.show()

**観察**: Lasso係数は途中で0，Ridge係数は0に近づくが0にならない．

In [ ]:
lasso_w = Lasso(alpha=0.1, max_iter=10000).fit(X_scaled, y).coef_
ridge_w = Ridge(alpha=0.1).fit(X_scaled, y).coef_
assert np.sum(np.abs(lasso_w) < 1e-5) > 0, "Lasso should zero some coefficients"
assert np.all(np.abs(ridge_w) > 1e-5), "Ridge should not zero coefficients"
print(f"Lasso: {np.sum(np.abs(lasso_w) > 1e-5)}/10 nonzero")
print(f"Ridge: {np.sum(np.abs(ridge_w) > 1e-5)}/10 nonzero")

## S2. Group Lasso実装（ISTA）

ISTA近接勾配法でGroup Lassoを解く（[Notion §5](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-4)）．10電極×4帯域，電極0,1だけ効くデータで，グループごと0になることを確認．

In [ ]:
X_g, y_g, w_true_g, groups = make_grouped_data()
X_g_scaled = StandardScaler().fit_transform(X_g)
print(f"Data: 200 samples, 10 electrodes x 4 bands = 40 features")
print(f"True active electrodes: 0, 1")

In [ ]:
w_glasso = group_lasso_ista(X_g_scaled, y_g, groups, lam=0.1)
active_elec = active_groups(w_glasso, groups)
print(f"Active electrodes: {active_elec}")
print(f"Nonzero coefficients: {np.sum(np.abs(w_glasso) > 1e-8)}/40")

**観察**: 電極0, 1が選ばれ，他8電極は全係数0（グループ単位で選択）．

In [ ]:
assert set(active_elec) == {0, 1}, "Should select electrodes 0 and 1"
for e in range(2, 10):
    assert np.all(np.abs(w_glasso[groups == e]) < 1e-8), f"Electrode {e} should be fully zero"

## S3. Lasso vs Group Lasso経路比較

Lassoは係数が個別に入り，Group Lassoは同じ電極の係数が同時に入る（[Notion §3](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-2)）．

In [ ]:
lambda_max_lasso = np.max(np.abs(X_g_scaled.T @ y_g)) / len(y_g)
lambda_max_glasso = max([np.linalg.norm(X_g_scaled[:, groups == g].T @ y_g) / (len(y_g) * np.sqrt(4)) 
                          for g in range(10)])
lambdas_lasso = np.logspace(np.log10(lambda_max_lasso), -3, 25)
lambdas_glasso = np.logspace(np.log10(lambda_max_glasso), -3, 25)

In [ ]:
lasso_path = np.array([Lasso(alpha=lam, max_iter=10000).fit(X_g_scaled, y_g).coef_ 
                        for lam in lambdas_lasso])
glasso_path = np.array([group_lasso_ista(X_g_scaled, y_g, groups, lam=lam) 
                         for lam in lambdas_glasso])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
colors = plt.cm.tab10(np.arange(10))
for i in range(40):
    elec = i // 4
    axes[0].plot(lambdas_lasso, lasso_path[:, i], color=colors[elec], alpha=0.6, linewidth=1)
    axes[1].plot(lambdas_glasso, glasso_path[:, i], color=colors[elec], alpha=0.6, linewidth=1)
for ax in axes:
    ax.set_xscale('log')
    ax.invert_xaxis()
    ax.axhline(0, color='k', linestyle='--', linewidth=0.5)
    ax.set_xlabel('lambda')
    ax.set_ylabel('Coefficient')
    ax.grid(True, alpha=0.3)
axes[0].set_title('Lasso path')
axes[1].set_title('Group Lasso path')
plt.tight_layout()
plt.show()

**観察**: Group Lassoは同色（同電極）の線が同じλで立ち上がる．Lassoは個別に入る．

## S4. 交差検証でλ選択

5-fold CVで検証誤差が最小になるλを見つける（[Notion §6](https://app.notion.com/p/3f43834a206d81d98a78c761f713daea#142-5)）．

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_mse_lasso = []
cv_mse_glasso = []
for lam_l, lam_g in zip(lambdas_lasso, lambdas_glasso):
    err_l, err_g = [], []
    for train_idx, val_idx in kf.split(X_g_scaled):
        Xtr, Xv = X_g_scaled[train_idx], X_g_scaled[val_idx]
        ytr, yv = y_g[train_idx], y_g[val_idx]
        w_l = Lasso(alpha=lam_l, max_iter=10000).fit(Xtr, ytr).coef_
        w_g = group_lasso_ista(Xtr, ytr, groups, lam=lam_g)
        err_l.append(mean_squared_error(yv, Xv @ w_l))
        err_g.append(mean_squared_error(yv, Xv @ w_g))
    cv_mse_lasso.append(np.mean(err_l))
    cv_mse_glasso.append(np.mean(err_g))
cv_mse_lasso = np.array(cv_mse_lasso)
cv_mse_glasso = np.array(cv_mse_glasso)

In [ ]:
best_lasso = np.argmin(cv_mse_lasso)
best_glasso = np.argmin(cv_mse_glasso)
print(f"Best lambda (Lasso): {lambdas_lasso[best_lasso]:.4f}, CV MSE: {cv_mse_lasso[best_lasso]:.4f}")
print(f"Best lambda (Group Lasso): {lambdas_glasso[best_glasso]:.4f}, CV MSE: {cv_mse_glasso[best_glasso]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(lambdas_lasso, cv_mse_lasso, 'o-', label='Lasso', alpha=0.7)
ax.plot(lambdas_glasso, cv_mse_glasso, 's-', label='Group Lasso', alpha=0.7)
ax.axvline(lambdas_lasso[best_lasso], color='C0', linestyle='--', alpha=0.5)
ax.axvline(lambdas_glasso[best_glasso], color='C1', linestyle='--', alpha=0.5)
ax.set_xscale('log')
ax.set_xlabel('lambda')
ax.set_ylabel('CV Mean Squared Error')
ax.set_title('Cross-validation')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**観察**: CV MSEが最小になるλが存在し，破線で示される．

In [ ]:
assert cv_mse_lasso[best_lasso] < cv_mse_lasso[0], "Best lambda should reduce CV error"
assert cv_mse_glasso[best_glasso] < cv_mse_glasso[0], "Best lambda should reduce CV error"

## まとめ

Lassoは係数を0にし，Group LassoはグループごとISTAで解ける．CVでλを選び，ECoGの電極・帯域選択に応用可能．